<a href="https://colab.research.google.com/github/marwa-nassane0052/quantum_ai_experiment/blob/main/test7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

folder_path = '/content/drive/MyDrive/skin_cancer_processed_dataset'

if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Folder '{folder_path}' not found. Please check the path and ensure it's in your My Drive.")

Contents of '/content/drive/MyDrive/skin_cancer_processed_dataset':
X_test.npy
y_test.npy
y_train.npy
X_train.npy


In [3]:
import numpy as np
folder_path = '/content/drive/MyDrive/skin_cancer_processed_dataset'
X_train = np.load(os.path.join(folder_path, 'X_train.npy'))
y_train = np.load(os.path.join(folder_path, 'y_train.npy'))
X_test = np.load(os.path.join(folder_path, 'X_test.npy'))
y_test = np.load(os.path.join(folder_path, 'y_test.npy'))

In [4]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2200, 3, 32, 32)
y_train shape: (2200,)
X_test shape: (660, 3, 32, 32)
y_test shape: (660,)


**Bulding the hybrid model**

In [5]:
!pip install pennylane

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# ── Hyperparameters ────────────────────────────────────────────────────────────
PATCH_SIZE  = 8       # 32/8 = 4 → 4×4 = 16 patches per image
STRIDE      = 8       # non-overlapping: zero redundant circuit calls
N_QUBITS    = 4
N_Q_LAYERS  = 1
BATCH_SIZE  = 16      # safe to use 16 now — only 256 circuit calls/batch
IN_CH       = 3

# ── Convert dataset to tensors ─────────────────────────────────────────────────
# data is already (N, 3, 32, 32) — no resize or permute needed
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

patches_per_img = ((32 - PATCH_SIZE) // STRIDE + 1) ** 2
print(f"Patches / image       : {patches_per_img}")          # 16
print(f"Circuit calls / batch : {patches_per_img * BATCH_SIZE}")  # 256

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),
                          batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)

# ── Quantum device ─────────────────────────────────────────────────────────────
try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    print("Using lightning.qubit  ✓")
except Exception:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    print("Using default.qubit (fallback)")

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (N_Q_LAYERS, N_QUBITS, 3)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

# ── Quantum Filter ─────────────────────────────────────────────────────────────
class QuantumFilter(nn.Module):
    """
    Slides a quantum circuit over 8×8 non-overlapping patches.
    Input  : (B, 3, 32, 32)
    Output : (B, 4,  4,  4)   ← 4 qubits × 4×4 patch grid
    """
    def __init__(self):
        super().__init__()
        patch_dim  = IN_CH * PATCH_SIZE * PATCH_SIZE   # 3×8×8 = 192
        self.proj  = nn.Linear(patch_dim, N_QUBITS, bias=False)
        self.qlayer = qlayer

    def forward(self, x):
        B, C, H, W = x.shape
        p, s = PATCH_SIZE, STRIDE

        # ── Extract patches ────────────────────────────────────────────────────
        patches = F.unfold(x, kernel_size=p, stride=s)   # (B, 192, 16)
        L       = patches.shape[2]                        # 16 patches
        patches = patches.permute(0, 2, 1).reshape(B * L, -1)  # (B*16, 192)

        # ── Project + normalise to [-π, π] ────────────────────────────────────
        q_in = torch.tanh(self.proj(patches)) * torch.pi  # (B*16, 4)

        # ── Quantum circuit evaluation ─────────────────────────────────────────
        q_out = torch.stack([self.qlayer(q_in[i]) for i in range(q_in.shape[0])])
        # (B*16, 4)

        # ── Reshape back to spatial feature map ───────────────────────────────
        H_out = (H - p) // s + 1   # = 4
        W_out = (W - p) // s + 1   # = 4
        q_out = q_out.reshape(B, L, N_QUBITS).permute(0, 2, 1)
        q_out = q_out.reshape(B, N_QUBITS, H_out, W_out)  # (B, 4, 4, 4)
        return q_out


# ── Classical CNN Classifier (3 conv layers) ───────────────────────────────────
class ClassicalCNN(nn.Module):
    """
    Input  : (B, 4, 4, 4)   — quantum feature map
    Output : (B, 2)          — class logits
    """
    def __init__(self):
        super().__init__()
        # padding=1 keeps spatial size stable before pooling
        self.conv1 = nn.Conv2d(N_QUBITS, 32,  kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32,       64,  kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64,       128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)

        # No MaxPool — feature map is already 4×4, pooling would collapse it
        # AdaptiveAvgPool to 2×2 gives a safe fixed size for the FC layer
        self.global_pool = nn.AdaptiveAvgPool2d((2, 2))
        self.dropout     = nn.Dropout(0.4)
        self.fc          = nn.Linear(128 * 2 * 2, 2)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))   # (B, 32, 4, 4)
        x = F.relu(self.bn2(self.conv2(x)))   # (B, 64, 4, 4)
        x = F.relu(self.bn3(self.conv3(x)))   # (B,128, 4, 4)
        x = self.global_pool(x)               # (B,128, 2, 2)
        x = self.dropout(x.reshape(x.size(0), -1)) # Changed .view to .reshape
        return self.fc(x)                     # (B, 2)


# ── Full Hybrid Model ──────────────────────────────────────────────────────────
class HybridQuantumCNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.quantum_filter = QuantumFilter()
        self.cnn            = ClassicalCNN()

    def forward(self, x):
        return self.cnn(self.quantum_filter(x))


# ── Initialise ─────────────────────────────────────────────────────────────────
model     = HybridQuantumCNNModel()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Sanity check shapes before committing to a full training run
with torch.no_grad():
    dummy  = X_train_t[:2]
    feat   = model.quantum_filter(dummy)
    logits = model(dummy)
    print(f"Quantum feature map : {feat.shape}")    # (2, 4, 4, 4)
    print(f"Model output logits : {logits.shape}")  # (2, 2)

# ── Training loop ──────────────────────────────────────────────────────────────
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, pred  = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (pred == labels).sum().item()

    scheduler.step()
    print(f"Epoch {epoch+1:02d}/{epochs} | "
          f"Loss: {running_loss/len(train_loader):.4f} | "
          f"Train Acc: {100*correct/total:.2f}%")

# ── Evaluation ─────────────────────────────────────────────────────────────────
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        _, pred  = torch.max(model(images), 1)
        total   += labels.size(0)
        correct += (pred == labels).sum().item()

print(f"\nTest Accuracy: {100 * correct / total:.2f}%")

Patches / image       : 16
Circuit calls / batch : 256
Using lightning.qubit  ✓
Quantum feature map : torch.Size([2, 4, 4, 4])
Model output logits : torch.Size([2, 2])


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 01/10 | Loss: 0.5086 | Train Acc: 75.18%
Epoch 02/10 | Loss: 0.4384 | Train Acc: 79.73%
Epoch 03/10 | Loss: 0.4134 | Train Acc: 80.64%
Epoch 04/10 | Loss: 0.3832 | Train Acc: 82.05%
Epoch 05/10 | Loss: 0.3651 | Train Acc: 82.95%
Epoch 06/10 | Loss: 0.3389 | Train Acc: 84.68%
Epoch 07/10 | Loss: 0.3217 | Train Acc: 85.59%
Epoch 08/10 | Loss: 0.3009 | Train Acc: 86.00%
Epoch 09/10 | Loss: 0.2990 | Train Acc: 86.95%
Epoch 10/10 | Loss: 0.2910 | Train Acc: 86.73%

Test Accuracy: 83.03%
